# Task 2 — Incremental CPG Parser Service

## Yêu cầu của đề

Parser xử lý từng file trong bounded memory, trích AST, CFG, DFG và call edge,
gán ID ổn định và phát structured event vào Kafka.

## Cách triển khai và lý do

[`task2/cpg_parser.py`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task2/cpg_parser.py) dùng standard-library
`ast`, structural path và SHA-256. [`task2/parser_service.py`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task2/parser_service.py)
đọc manifest theo dòng, chỉ giữ một file trong bộ nhớ, phát UPSERT/DELETE và chỉ
lưu state sau khi producer đã `flush`. State của revision trước cho phép dọn
node/edge stale khi nội dung thay đổi.

Hợp đồng dùng thống nhất `schema_version="1.0.0"`, `path`,
`content_sha256`, `source_id`/`target_id` và edge type `CALL`.

## Output thật đã chạy

Cell này đọc summary của parser và kết quả kiểm tra toàn corpus. Script kiểm tra
đếm distinct ID và toàn bộ endpoint edge.


In [1]:
import json
from pathlib import Path

summary = json.loads(Path("../artifacts/task2/summary.json").read_text(encoding="utf-8"))
proof = json.loads(Path("../artifacts/task2/corpus_verification.json").read_text(encoding="utf-8"))
schema = json.loads(Path("../artifacts/task2/schema_validation.json").read_text(encoding="utf-8"))
print(json.dumps({
    "targeted_files": summary["total_files_targeted"],
    "processed_files": summary["processed_files"],
    "parser_errors": summary["error_files"],
    "node_events": proof["node_events"],
    "distinct_node_ids": proof["distinct_node_ids"],
    "edge_events": proof["edge_events"],
    "distinct_edge_ids": proof["distinct_edge_ids"],
    "metadata_events": proof["metadata_events"],
    "distinct_metadata_file_ids": proof["distinct_metadata_file_ids"],
    "missing_edge_sources": proof["missing_edge_sources"],
    "missing_edge_targets": proof["missing_edge_targets"],
    "schema_validation": schema,
}, indent=2))


{
  "targeted_files": 147,
  "processed_files": 147,
  "parser_errors": 0,
  "node_events": 208003,
  "distinct_node_ids": 208003,
  "edge_events": 267695,
  "distinct_edge_ids": 267695,
  "metadata_events": 147,
  "distinct_metadata_file_ids": 147,
  "missing_edge_sources": 0,
  "missing_edge_targets": 0,
  "schema_validation": {
    "status": "PASS",
    "command": "python -m unittest tests.test_event_contract -v",
    "source": "Task 2 publish_success, replay DELETE, and publish_failure events",
    "schemas": [
      "node-event.schema.json",
      "edge-event.schema.json",
      "metadata-event.schema.json",
      "error-event.schema.json"
    ],
    "tests": 2
  }
}


## Code/lệnh quan trọng

```bash
python task2/parser_service.py --manifest artifacts/task1/python_manifest.jsonl \
  --repo-dir .work/repos/datasets --kafka-bootstrap localhost:9092
python task2/verify_corpus.py artifacts/task2 --expected-files 147
```

Source liên quan:
[`event_contract.py`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task2/event_contract.py),
[`parser_state.py`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task2/parser_state.py),
[`verify_corpus.py`](https://github.com/linh285/huggingface-cpg-streaming/blob/fix/lab04-final-integration/task2/verify_corpus.py).

## Lỗi đã gặp và cách khắc phục

Contract cũ từng lệch giữa `CALLS`/`CALL`, `file_path`/`path` và kiểu
`schema_version`. Nhóm bỏ lớp tương thích mơ hồ, dùng một contract duy nhất và
thêm test schema bằng event thật. Kiểm tra chỉ dựa vào tổng count cũng chưa bắt
được ID trùng hoặc endpoint thiếu, nên regression cuối so sánh total với
distinct và kiểm từng `source_id`, `target_id`.

## Reflection

Structural path ổn định hơn số dòng cho replay không đổi; state cũ cộng DELETE
event mới là phần cần thiết để file sửa không để topology stale.

## Tái lập

Có thể thêm `--dry-run --output-dir artifacts/task2` để sinh JSONL và chạy
corpus verifier mà không cần Kafka.
